# 01 — Exploration Notebook

* Purpose : Use this notebook to test individual pipeline components before wiringcthem together in graph.py. 

* Author : Gina Nguyen


## 0. Setup

In [1]:
import sys
sys.path.insert(0, "..")   # make src/ importable from notebooks/

from dotenv import load_dotenv
load_dotenv("../.env")

print("Environment loaded ✓")

Environment loaded ✓


## 1. Inspect the dataset

In [2]:
import pandas as pd

df = pd.read_csv("../data/raw_companies.csv")
print(f"Shape: {df.shape}")
df.head(10)

# %%
# How many of each difficulty level?
df["difficulty"].value_counts()

Shape: (100, 5)


difficulty
medium    40
easy      33
hard      27
Name: count, dtype: int64

## 2. Test fuzzy matching (no API key needed)

In [3]:
import json
from src.agent.tools import fuzzy_match_tool, company_lookup_tool

test_names = ["Aple Inc", "MSFT", "Saleforce.com", "WandB", "Figma Design"]

for name in test_names:
    result = json.loads(fuzzy_match_tool.invoke(name))
    status = "✓" if result["match_found"] else "✗"
    canonical = result.get("canonical_name", "—")
    score = result.get("similarity_score", "—")
    print(f"{status} {name:25} → {canonical:30} (score: {score})")

✓ Aple Inc                  → Apple Inc.                     (score: 88.88888888888889)
✗ MSFT                      → —                              (score: —)
✗ Saleforce.com             → —                              (score: —)
✗ WandB                     → —                              (score: —)
✗ Figma Design              → —                              (score: —)


## 3. Test a single company resolution (needs OPENAI_API_KEY)

In [4]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [5]:
from src.agent.resolver import resolve_company

result = resolve_company("Aple Inc")
print(result)

{'canonical_name': 'Apple Inc.', 'domain': 'apple.com', 'confidence': 0.9, 'evidence': "Fuzzy matched 'Aple Inc' → 'Apple Inc.' (score 88.88888888888889/100)"}


## 4. Test the LLM-as-judge 

In [6]:
from src.evaluator.judge import evaluate_match

verdict = evaluate_match(
    raw_name="Aple Inc",
    resolved_name="Apple Inc.",
    domain="apple.com",
    evidence="Fuzzy matched with score 92/100",
    confidence=0.95,
)

print(f"Score:    {verdict.score}/10")
print(f"Verdict:  {verdict.verdict}")
print(f"Reason:   {verdict.rationale}")

Score:    9/10
Verdict:  accept
Reason:   The resolved name 'Apple Inc.' is a strong match for the raw input 'Aple Inc' with a high fuzzy matching score of 92/100 and high confidence of 0.95, indicating a very good match with only a minor name variation.


## 5. Run pipeline on 5 rows (needs all API keys)

In [17]:
from src.pipeline.graph import run_pipeline

state = run_pipeline(input_path="../data/raw_companies.csv", limit=5)

# %%
# Inspect results
results_df = pd.DataFrame(state["records"])
results_df[["raw_name", "canonical_name", "confidence", "judge_score", "verdict"]]


Pipeline run 00b7e005 starting...

[ingest] Loaded 5 records from ../data/_tmp_limit_5.csv


Resolving:   0%|          | 0/1 [00:00<?, ?it/s]

Resolving: 100%|██████████| 1/1 [00:09<00:00,  9.50s/it]


[resolve] Resolved 0 / 5


Evaluating: 100%|██████████| 5/5 [00:00<00:00, 58743.75it/s]

[evaluate] accept=0  review=0  reject=5
[export] Wrote 5 rows → data/resolved_output.csv

── Run summary ──────────────────────────────────
  Total:    5
  Accepted: 0
  Review:   0
  Rejected: 5
  Run ID:   00b7e005  (search in Langfuse)
─────────────────────────────────────────────────



,raw_name,canonical_name,confidence,judge_score,verdict
0,Apple Inc.,None,0.0,0,reject
1,Aple Inc,None,0.0,0,reject
2,APPLE,None,0.0,0,reject
3,Apple Incorporated,None,0.0,0,reject
4,Microsoft Corp,None,0.0,0,reject


## 6. Human labeling

In [8]:
# TODO: paste 20 records from resolved_output.csv here and add your labels
# human_labels = {
#     1:  {"your_verdict": "accept", "your_score": 9},
#     2:  {"your_verdict": "review", "your_score": 5},
#     ...
# }
#
# Then compute agreement:
# agreement = sum(1 for id, h in human_labels.items()
#                 if h["your_verdict"] == results_df.loc[id, "verdict"])
# print(f"Judge agrees with you on {agreement}/20 = {agreement/20:.0%}")

In [15]:
# state = run_pipeline(input_path="../data/raw_companies.csv", limit=20)

# # %%
# # Inspect results
# results_df = pd.DataFrame(state["records"])
# results_df[["raw_name", "canonical_name", "confidence", "judge_score", "verdict"]]


Pipeline run 6e560e50 starting...

[ingest] Loaded 20 records from ../data/_tmp_limit_20.csv


Resolving:   0%|          | 0/2 [00:00<?, ?it/s]

Resolving: 100%|██████████| 2/2 [00:03<00:00,  1.77s/it]


[resolve] Resolved 0 / 20


Evaluating: 100%|██████████| 20/20 [00:00<00:00, 139810.13it/s]

[evaluate] accept=0  review=0  reject=20
[export] Wrote 20 rows → data/resolved_output.csv

── Run summary ──────────────────────────────────
  Total:    20
  Accepted: 0
  Review:   0
  Rejected: 20
  Run ID:   6e560e50  (search in Langfuse)
─────────────────────────────────────────────────



,raw_name,canonical_name,confidence,judge_score,verdict
0,Apple Inc.,None,0.0,0,reject
1,Aple Inc,None,0.0,0,reject
2,APPLE,None,0.0,0,reject
3,Apple Incorporated,None,0.0,0,reject
4,Microsoft Corp,None,0.0,0,reject
5,Microsft Corporation,None,0.0,0,reject
6,MSFT,None,0.0,0,reject
7,microsoft,None,0.0,0,reject
8,Alphabet Inc,None,0.0,0,reject
9,Google (Alphabet),None,0.0,0,reject


In [20]:
# human_labels = {
#     1:  {"your_verdict": "accept", "your_score": 9},
#     2:  {"your_verdict": "review", "your_score": 5},
#     3:  {"your_verdict": "review", "your_score": 5},
#     4:  {"your_verdict": "review", "your_score": 5},
#     5:  {"your_verdict": "review", "your_score": 5},
#     6:  {"your_verdict": "review", "your_score": 5},
#     7:  {"your_verdict": "review", "your_score": 5},
#     8:  {"your_verdict": "review", "your_score": 5},
#     9:  {"your_verdict": "review", "your_score": 5},
#     10:  {"your_verdict": "review", "your_score": 5},
#     11:  {"your_verdict": "accept", "your_score": 9},
#     12:  {"your_verdict": "review", "your_score": 5},
#     13:  {"your_verdict": "review", "your_score": 5},
#     14:  {"your_verdict": "review", "your_score": 5},
#     15:  {"your_verdict": "review", "your_score": 5},
#     16:  {"your_verdict": "review", "your_score": 5},
#     17:  {"your_verdict": "review", "your_score": 5},
#     18:  {"your_verdict": "review", "your_score": 5},
#     19:  {"your_verdict": "review", "your_score": 5},
#     20:  {"your_verdict": "review", "your_score": 5},
# }

human_labels = {
    0:  {"your_verdict": "accept", "your_score": 10},
    1:  {"your_verdict": "accept", "your_score": 10},
    2:  {"your_verdict": "reject", "your_score": 0},
    3:  {"your_verdict": "reject", "your_score": 0},
    4:  {"your_verdict": "accept", "your_score": 10}
}

In [21]:
agreement = sum(1 for id, h in human_labels.items()
                if h["your_verdict"] == results_df.loc[id, "verdict"])
print(f"Judge agrees with you on {agreement}/5 = {agreement/5:.0%}")

Judge agrees with you on 2/5 = 40%
